# 🍽️ Restaurant Data Analysis — Exploratory Data Analysis (EDA)

**Author:** Siddharth Pandey  
**Dataset:** Zomato Restaurant Dataset — 9,500+ restaurants across 15 countries  
**Goal:** Clean the dataset, identify top revenue-driving cuisine categories, and extract actionable business insights.

---

## 📌 Table of Contents
1. [Import Libraries](#1)
2. [Load & Inspect Data](#2)
3. [Data Cleaning](#3)
4. [Univariate Analysis](#4)
5. [Cuisine Analysis & Revenue Drivers](#5)
6. [Geographic Analysis](#6)
7. [Rating & Customer Engagement](#7)
8. [Pricing & Service Analysis](#8)
9. [Key Business Insights Summary](#9)

## 1. Import Libraries <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Style ──
plt.rcParams.update({
    'figure.facecolor': '#0e0e16',
    'axes.facecolor':   '#16161f',
    'axes.edgecolor':   '#2a2a3a',
    'axes.labelcolor':  '#9495a7',
    'axes.titlecolor':  '#f0f0f5',
    'xtick.color':      '#9495a7',
    'ytick.color':      '#9495a7',
    'text.color':       '#f0f0f5',
    'grid.color':       '#2a2a3a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'figure.titlesize': 16,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})

PALETTE = ['#6366f1','#22d3ee','#34d399','#fbbf24','#f43f5e',
           '#a855f7','#fb923c','#3b82f6','#14b8a6','#ec4899']

print('✅ Libraries loaded successfully!')

## 2. Load & Inspect Data <a id='2'></a>

In [ ]:
df = pd.read_csv('../DataSet/Dataset .csv', encoding='utf-8', on_bad_lines='skip')
df.columns = [c.strip() for c in df.columns]

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
print('=== Column Info ===')
df.info()

In [ ]:
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct}).query('`Missing Count` > 0')

In [ ]:
print('=== Descriptive Statistics ===')
df.describe().round(2)

## 3. Data Cleaning <a id='3'></a>

In [ ]:
# ── Country mapping ──
COUNTRY_MAP = {
    1:'India', 14:'Australia', 30:'Brazil', 37:'Canada',
    94:'Indonesia', 148:'New Zealand', 162:'Philippines',
    166:'Qatar', 184:'Singapore', 189:'South Africa',
    191:'Sri Lanka', 208:'Turkey', 214:'UAE',
    215:'United Kingdom', 216:'United States'
}
df['Country'] = df['Country Code'].map(COUNTRY_MAP).fillna('Unknown')

# ── Type corrections ──
df['Aggregate rating']       = pd.to_numeric(df['Aggregate rating'], errors='coerce').fillna(0)
df['Votes']                  = pd.to_numeric(df['Votes'], errors='coerce').fillna(0).astype(int)
df['Average Cost for two']   = pd.to_numeric(df['Average Cost for two'], errors='coerce').fillna(0)
df['Price range']            = pd.to_numeric(df['Price range'], errors='coerce').fillna(0).astype(int)

# ── Boolean ──
df['Has Table booking']   = df['Has Table booking'].str.strip().eq('Yes')
df['Has Online delivery'] = df['Has Online delivery'].str.strip().eq('Yes')

# ── Fill nulls ──
df['Cuisines']         = df['Cuisines'].fillna('Unknown').str.strip()
df['Restaurant Name']  = df['Restaurant Name'].str.strip()
df['City']             = df['City'].str.strip()

# ── Price labels ──
df['Price Label'] = df['Price range'].map(
    {1:'Budget ($)', 2:'Mid-Range ($$)', 3:'Premium ($$$)', 4:'Luxury ($$$$)'}
).fillna('Unknown')

# ── Rating categories ──
def rate_cat(r):
    if r == 0:   return 'Not Rated'
    elif r < 2:  return 'Poor'
    elif r < 3:  return 'Average'
    elif r < 3.5: return 'Good'
    elif r < 4:  return 'Very Good'
    elif r < 4.5: return 'Excellent'
    else:         return 'Outstanding'
df['Rating Category'] = df['Aggregate rating'].apply(rate_cat)

# ── Revenue index ──
df['Revenue Index'] = df['Average Cost for two'] * df['Votes'].clip(lower=1)

# ── Remove duplicates ──
before = len(df)
df = df.drop_duplicates(subset=['Restaurant ID'])
after = len(df)

print(f'✅ Data cleaning complete!')
print(f'   Rows before: {before:,}  |  After: {after:,}  |  Removed: {before-after:,} duplicates')
print(f'   Countries mapped: {df["Country"].nunique()}')
print(f'   Cities: {df["City"].nunique()}')
df.head(3)

## 4. Univariate Analysis <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Univariate Analysis — Key Numeric Columns', fontsize=15, fontweight='bold', color='#f0f0f5')

# Rating distribution
rated = df[df['Aggregate rating'] > 0]
axes[0].hist(rated['Aggregate rating'], bins=20, color='#6366f1', edgecolor='#0e0e16', alpha=0.85)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Aggregate Rating')
axes[0].set_ylabel('Count')
axes[0].axvline(rated['Aggregate rating'].mean(), color='#22d3ee', linestyle='--',
                linewidth=2, label=f'Mean: {rated["Aggregate rating"].mean():.2f}')
axes[0].legend()
axes[0].grid(axis='y')

# Votes distribution (log scale for clarity)
voted = df[df['Votes'] > 0]
axes[1].hist(np.log1p(voted['Votes']), bins=25, color='#34d399', edgecolor='#0e0e16', alpha=0.85)
axes[1].set_title('Votes Distribution (log scale)')
axes[1].set_xlabel('log(Votes + 1)')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y')

# Price range
pr_counts = df['Price Label'].value_counts()
pr_colors = ['#34d399','#3b82f6','#fbbf24','#f43f5e']
axes[2].bar(pr_counts.index, pr_counts.values,
            color=pr_colors[:len(pr_counts)], edgecolor='#0e0e16')
axes[2].set_title('Price Range Distribution')
axes[2].set_xlabel('Price Range')
axes[2].set_ylabel('Count')
for i, v in enumerate(pr_counts.values):
    axes[2].text(i, v + 30, f'{v:,}', ha='center', va='bottom', fontsize=9, color='#f0f0f5')
axes[2].grid(axis='y')

plt.tight_layout()
plt.savefig('charts/01_univariate.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()
print('📊 Chart saved to charts/01_univariate.png')

## 5. Cuisine Analysis & Top Revenue Drivers <a id='5'></a>

In [ ]:
# ── Explode multi-cuisine column ──
cuisine_rows = []
for _, row in df.iterrows():
    for c in str(row['Cuisines']).split(','):
        c = c.strip()
        if c and c != 'Unknown':
            cuisine_rows.append({
                'Cuisine': c,
                'Revenue Index': row['Revenue Index'],
                'Votes': row['Votes'],
                'Rating': row['Aggregate rating'],
                'Cost': row['Average Cost for two'],
                'Country': row['Country']
            })

cuisine_df = pd.DataFrame(cuisine_rows)
print(f'✅ Unique cuisines found: {cuisine_df["Cuisine"].nunique()}')

# ── Top 15 cuisines by count ──
top15 = cuisine_df['Cuisine'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top15.index[::-1], top15.values[::-1], color=PALETTE[:15], edgecolor='#0e0e16')
ax.set_title('Top 15 Most Popular Cuisines', fontweight='bold')
ax.set_xlabel('Number of Restaurants')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.3)
for bar in bars:
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9, color='#9495a7')
plt.tight_layout()
plt.savefig('charts/02_top_cuisines.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

In [ ]:
# ── Revenue-driving categories ──
revenue_by_cuisine = (
    cuisine_df.groupby('Cuisine')
    .agg(
        Total_Revenue = ('Revenue Index', 'sum'),
        Restaurant_Count = ('Cuisine', 'count'),
        Avg_Rating = ('Rating', 'mean'),
        Avg_Cost   = ('Cost', 'mean'),
        Total_Votes = ('Votes', 'sum')
    )
    .reset_index()
    .query('Restaurant_Count >= 10')
    .sort_values('Total_Revenue', ascending=False)
)

print('🏆 TOP 10 REVENUE-DRIVING CUISINE CATEGORIES')
print('=' * 60)
display = revenue_by_cuisine.head(10).copy()
display['Total_Revenue'] = display['Total_Revenue'].apply(lambda x: f'{x:,.0f}')
display['Total_Votes']   = display['Total_Votes'].apply(lambda x: f'{x:,}')
display['Avg_Rating']    = display['Avg_Rating'].round(2)
display['Avg_Cost']      = display['Avg_Cost'].round(0).astype(int)
print(display.to_string(index=False))

In [ ]:
# ── Plot Top 3 Revenue Drivers ──
top3_rev = revenue_by_cuisine.head(3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('🏆 Top 3 Revenue-Driving Cuisine Categories', fontsize=15, fontweight='bold', color='#f0f0f5')

medals = ['🥇', '🥈', '🥉']
colors = ['#6366f1', '#f43f5e', '#fbbf24']

for i, (_, row) in enumerate(top3_rev.iterrows()):
    ax = axes[i]
    metrics = ['Restaurant\nCount', 'Avg\nRating', 'Total\nVotes (K)']
    values  = [row['Restaurant_Count'], row['Avg_Rating'], row['Total_Votes']/1000]
    bars = ax.bar(metrics, values, color=colors[i], alpha=0.85, edgecolor='#0e0e16', width=0.5)
    ax.set_title(f'{medals[i]} {row["Cuisine"]}', fontweight='bold', fontsize=12)
    ax.set_ylim(0, max(values) * 1.25)
    for bar, val in zip(bars, [row['Restaurant_Count'], row['Avg_Rating'], row['Total_Votes']/1000]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f'{val:,.1f}', ha='center', va='bottom', fontsize=10, color='#f0f0f5', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    ax.set_facecolor('#16161f')

plt.tight_layout()
plt.savefig('charts/03_top3_revenue.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

## 6. Geographic Analysis <a id='6'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Geographic Distribution', fontsize=15, fontweight='bold', color='#f0f0f5')

# Country distribution
country_counts = df['Country'].value_counts()
axes[0].barh(country_counts.index[::-1], country_counts.values[::-1],
             color=[PALETTE[i % len(PALETTE)] for i in range(len(country_counts))],
             edgecolor='#0e0e16')
axes[0].set_title('Restaurants by Country')
axes[0].set_xlabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].grid(axis='x', alpha=0.3)

# Top 10 cities
city_counts = df['City'].value_counts().head(10)
axes[1].bar(city_counts.index, city_counts.values,
            color=PALETTE[:10], edgecolor='#0e0e16')
axes[1].set_title('Top 10 Cities by Restaurant Count')
axes[1].set_xlabel('City')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=40)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('charts/04_geographic.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

## 7. Rating & Customer Engagement <a id='7'></a>

In [ ]:
rated = df[df['Aggregate rating'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Rating & Customer Engagement', fontsize=15, fontweight='bold', color='#f0f0f5')

# Rating category donut
cat_order  = ['Outstanding','Excellent','Very Good','Good','Average','Poor','Not Rated']
cat_colors = ['#6366f1','#22d3ee','#34d399','#fbbf24','#fb923c','#f43f5e','#5e5f72']
cat_counts = df['Rating Category'].value_counts().reindex(cat_order, fill_value=0)
wedges, texts, autotexts = axes[0].pie(
    cat_counts.values, labels=cat_counts.index,
    colors=cat_colors, autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0e0e16'),
    textprops=dict(color='#9495a7', fontsize=9)
)
for at in autotexts: at.set_color('#f0f0f5'); at.set_fontsize(8)
axes[0].set_title('Rating Category Distribution')

# Scatter — votes vs rating (colored by price range)
sample = rated[rated['Votes'] > 0].sample(min(800, len(rated)), random_state=42)
price_palette = {'Budget ($)': '#34d399', 'Mid-Range ($$)': '#3b82f6',
                 'Premium ($$$)': '#fbbf24', 'Luxury ($$$$)': '#f43f5e'}
for pr, grp in sample.groupby('Price Label'):
    axes[1].scatter(grp['Votes'], grp['Aggregate rating'],
                    c=price_palette.get(pr, '#6366f1'), alpha=0.55, s=18, label=pr)
axes[1].set_title('Customer Votes vs Rating')
axes[1].set_xlabel('Number of Votes')
axes[1].set_ylabel('Aggregate Rating')
axes[1].legend(fontsize=8, title='Price Range', title_fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('charts/05_rating_engagement.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

print(f'\n📊 Rating Stats (rated restaurants only):')
print(f'   Mean Rating  : {rated["Aggregate rating"].mean():.3f}')
print(f'   Median Rating: {rated["Aggregate rating"].median():.3f}')
print(f'   Std Dev      : {rated["Aggregate rating"].std():.3f}')
print(f'   Min / Max    : {rated["Aggregate rating"].min():.1f} / {rated["Aggregate rating"].max():.1f}')

## 8. Pricing & Service Analysis <a id='8'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Pricing & Digital Service Analysis', fontsize=15, fontweight='bold', color='#f0f0f5')

# Avg rating by price range
pr_stats = (rated.groupby('Price Label')
            .agg(Avg_Rating=('Aggregate rating','mean'), Count=('Restaurant Name','count'))
            .reset_index().sort_values('Price Label'))

bars = axes[0].bar(pr_stats['Price Label'], pr_stats['Avg_Rating'],
                   color=['#34d399','#3b82f6','#fbbf24','#f43f5e'], edgecolor='#0e0e16')
axes[0].set_title('Avg Rating by Price Range')
axes[0].set_xlabel('Price Range')
axes[0].set_ylabel('Average Rating')
axes[0].set_ylim(0, 5)
for bar, (_, row) in zip(bars, pr_stats.iterrows()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{row["Avg_Rating"]:.2f}\n(n={row["Count"]:,})',
                 ha='center', va='bottom', fontsize=9, color='#f0f0f5')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=15)

# Digital service adoption stacked bar
services = ['Online Delivery', 'Table Booking']
yes_vals = [df['Has Online delivery'].sum(), df['Has Table booking'].sum()]
no_vals  = [len(df) - v for v in yes_vals]
x = np.arange(len(services))
axes[1].bar(x, yes_vals, label='Available ✅', color='#34d399', edgecolor='#0e0e16')
axes[1].bar(x, no_vals, bottom=yes_vals, label='Not Available ❌',
            color='rgba(244,63,94,0.5)', edgecolor='#0e0e16')
axes[1].set_xticks(x)
axes[1].set_xticklabels(services)
axes[1].set_title('Digital Service Adoption')
axes[1].set_ylabel('Number of Restaurants')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
for xi, (y, n) in enumerate(zip(yes_vals, no_vals)):
    pct = y / (y + n) * 100
    axes[1].text(xi, y/2, f'{pct:.1f}%', ha='center', va='center',
                 fontsize=11, fontweight='bold', color='#0e0e16')

plt.tight_layout()
plt.savefig('charts/06_pricing_services.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

In [ ]:
# ── Correlation Heatmap ──
num_cols = ['Average Cost for two', 'Price range', 'Aggregate rating', 'Votes', 'Revenue Index']
corr = df[num_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5, linecolor='#0e0e16',
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Features', fontweight='bold')
ax.set_facecolor('#16161f')
plt.tight_layout()
plt.savefig('charts/07_correlation.png', dpi=150, bbox_inches='tight', facecolor='#0e0e16')
plt.show()

## 9. Key Business Insights Summary <a id='9'></a>

In [ ]:
top3_names = revenue_by_cuisine.head(3)['Cuisine'].tolist()

print('=' * 65)
print('  🍽️  RESTAURANT EDA — KEY BUSINESS INSIGHTS SUMMARY')
print('=' * 65)

print(f"""
📊 DATASET OVERVIEW
   Total Restaurants : {len(df):,}
   Countries Covered : {df['Country'].nunique()}
   Cities            : {df['City'].nunique()}
   Unique Cuisines   : {cuisine_df['Cuisine'].nunique()}
   Total Votes       : {df['Votes'].sum():,}

🏆 TOP 3 REVENUE-DRIVING CUISINE CATEGORIES
   🥇 {top3_names[0]}
   🥈 {top3_names[1]}
   🥉 {top3_names[2]}

⭐ QUALITY INSIGHTS
   Average Rating (rated only) : {rated['Aggregate rating'].mean():.3f} / 5.0
   Outstanding Restaurants     : {(rated['Aggregate rating'] >= 4.5).sum():,} ({(rated['Aggregate rating'] >= 4.5).mean()*100:.1f}%)
   Most Common Rating Category : {df['Rating Category'].value_counts().idxmax()}

📱 DIGITAL SERVICE ADOPTION GAPS
   Online Delivery Available : {df['Has Online delivery'].mean()*100:.1f}%
   Table Booking Available   : {df['Has Table booking'].mean()*100:.1f}%
   → Massive opportunity: 75%+ restaurants not on delivery platforms!

💰 PRICING INSIGHTS
   Luxury restaurants avg rating : {rated[rated['Price Label']=='Luxury ($$$$)']['Aggregate rating'].mean():.2f}
   Budget restaurants avg rating : {rated[rated['Price Label']=='Budget ($)']['Aggregate rating'].mean():.2f}
   → Premium pricing correlates with better customer satisfaction

🌍 MARKET CONCENTRATION
   Top Country : {df['Country'].value_counts().idxmax()} ({df['Country'].value_counts().iloc[0]/len(df)*100:.1f}% of all listings)
   Top City    : {df['City'].value_counts().idxmax()} ({df['City'].value_counts().iloc[0]:,} restaurants)
""")

print('=' * 65)
print('  ✅ Analysis Complete — Charts saved to charts/ folder')
print('=' * 65)